In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

BASE_DIR = Path.cwd().parent

RATINGS_PATH = BASE_DIR / "data" / "raw" / "ratings.csv"

ratings = pd.read_csv(RATINGS_PATH)

print("Ratings:", ratings.shape)
display(ratings.head())

Ratings: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", trainset.n_ratings)
print("Test ratings:", len(testset))

Training ratings: 80668
Test ratings: 20168


In [3]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD trained")

✅ SVD trained


In [4]:
predictions = svd_model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=False
)

mae = accuracy.mae(
    predictions,
    verbose=False
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8807
MAE:  0.6766


In [5]:
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    svd_model,
    MODEL_DIR / "svd_model.joblib"
)

print("✅ SVD model saved")

✅ SVD model saved


In [6]:
Get-ChildItem models

SyntaxError: invalid syntax (279386941.py, line 1)

In [7]:
weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9),
]

results = []

for content_weight, collaborative_weight in weights:
    results.append({
        "content_weight": content_weight,
        "collaborative_weight": collaborative_weight,
        "weight_sum": content_weight + collaborative_weight
    })

weights_df = pd.DataFrame(results)

display(weights_df)

,content_weight,collaborative_weight,weight_sum
0,0.9,0.1,1.0
1,0.8,0.2,1.0
2,0.7,0.3,1.0
3,0.6,0.4,1.0
4,0.5,0.5,1.0
5,0.4,0.6,1.0
6,0.3,0.7,1.0
7,0.2,0.8,1.0
8,0.1,0.9,1.0


In [8]:
K = 10
RELEVANCE_THRESHOLD = 4.0

print(f"K = {K}")
print(f"Relevant rating >= {RELEVANCE_THRESHOLD}")

K = 10
Relevant rating >= 4.0


In [9]:
def precision_recall_f1_at_k(
    recommended_ids,
    relevant_ids,
    k=10
):
    recommended_ids = list(recommended_ids)[:k]
    relevant_ids = set(relevant_ids)

    if not recommended_ids:
        return 0.0, 0.0, 0.0

    hits = len(
        set(recommended_ids) & relevant_ids
    )

    precision = hits / len(recommended_ids)

    recall = (
        hits / len(relevant_ids)
        if relevant_ids
        else 0.0
    )

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return precision, recall, f1

In [10]:
test_recommended = [1, 2, 3, 4, 5]
test_relevant = [2, 4, 8, 10]

precision, recall, f1 = precision_recall_f1_at_k(
    test_recommended,
    test_relevant,
    k=5
)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Precision: 0.4
Recall: 0.5
F1: 0.4444444444444445


In [11]:
# Build a lookup of relevant movies from the test set
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

test_df["relevant"] = (
    test_df["actual_rating"] >= RELEVANCE_THRESHOLD
)

relevant_by_user = (
    test_df[test_df["relevant"]]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Test users:", len(relevant_by_user))
print("Relevant interactions:", test_df["relevant"].sum())

Test users: 595
Relevant interactions: 9712


In [12]:
def hybrid_scores_for_user(
    user_id,
    content_weight,
    collaborative_weight
):
    # SVD scores
    svd_scores = []

    for movie_id in df["id"]:
        prediction = svd_model.predict(
            user_id,
            int(movie_id)
        )

        svd_scores.append(prediction.est)

    scores = pd.DataFrame({
        "movieId": df["id"].values,
        "content_score": np.zeros(len(df)),
        "collaborative_score": svd_scores
    })

    # Normalize collaborative scores
    scores["collaborative_score"] = min_max_normalize(
        scores["collaborative_score"]
    )

    return scores

In [13]:
def evaluate_svd_at_k(
    model,
    test_df,
    k=10,
    threshold=4.0
):
    precisions = []
    recalls = []
    f1_scores = []

    relevant_by_user = (
        test_df[test_df["actual_rating"] >= threshold]
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    for user_id, relevant_movies in relevant_by_user.items():

        candidate_movies = df["id"].tolist()

        predictions = [
            (
                movie_id,
                model.predict(
                    int(user_id),
                    int(movie_id)
                ).est
            )
            for movie_id in candidate_movies
        ]

        predictions.sort(
            key=lambda x: x[1],
            reverse=True
        )

        recommended = [
            movie_id
            for movie_id, _ in predictions[:k]
        ]

        precision, recall, f1 = (
            precision_recall_f1_at_k(
                recommended,
                relevant_movies,
                k
            )
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return {
        "precision@10": np.mean(precisions),
        "recall@10": np.mean(recalls),
        "f1@10": np.mean(f1_scores)
    }

In [14]:
svd_metrics = evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

print(svd_metrics)

NameError: name 'df' is not defined

In [15]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

BASE_DIR = Path.cwd().parent

# Load TMDB features
MOVIES_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "movies_features.csv"
)

df = pd.read_csv(MOVIES_PATH)

print("TMDB dataset:", df.shape)
display(df[["id", "title"]].head())

TMDB dataset: (4803, 11)


,id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter


In [16]:
RATINGS_PATH = BASE_DIR / "data" / "raw" / "ratings.csv"

ratings = pd.read_csv(RATINGS_PATH)

print("Ratings:", ratings.shape)
display(ratings.head())

Ratings: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [17]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", trainset.n_ratings)
print("Test ratings:", len(testset))

Training ratings: 80668
Test ratings: 20168


In [18]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD model ready")

✅ SVD model ready


In [19]:
predictions = svd_model.test(testset)

rmse = accuracy.rmse(
    predictions,
    verbose=False
)

mae = accuracy.mae(
    predictions,
    verbose=False
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.8807
MAE:  0.6766


In [20]:
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

print("Test dataframe:", test_df.shape)

display(test_df.head())

Test dataframe: (20168, 3)


,userId,movieId,actual_rating
0,140,6765,3.5
1,603,290,4.0
2,438,5055,4.0
3,433,164179,5.0
4,474,5114,4.0


In [21]:
K = 10
RELEVANCE_THRESHOLD = 4.0

In [22]:
evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

{'precision@10': np.float64(0.02453781512605042),
 'recall@10': np.float64(0.0254095509671597),
 'f1@10': np.float64(0.020357676182624025)}

In [23]:
df["id"]

0        19995
1          285
2       206647
3        49026
4        49529
         ...  
4798      9367
4799     72766
4800    231617
4801    126186
4802     25975
Name: id, Length: 4803, dtype: int64

In [24]:
LINKS_PATH = BASE_DIR / "data" / "raw" / "links.csv"

links = pd.read_csv(LINKS_PATH)

ratings_tmdb = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

ratings_tmdb = ratings_tmdb.dropna(
    subset=["tmdbId"]
).copy()

ratings_tmdb["tmdbId"] = (
    ratings_tmdb["tmdbId"]
    .astype(int)
)

In [25]:
ratings_tmdb = ratings_tmdb[
    ratings_tmdb["tmdbId"].isin(df["id"])
].copy()

print(
    "Ratings matching TMDB catalog:",
    len(ratings_tmdb)
)

Ratings matching TMDB catalog: 70194


In [26]:
svd_ratings = ratings_tmdb[
    ["userId", "tmdbId", "rating"]
].rename(
    columns={"tmdbId": "movieId"}
)

In [27]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data = Dataset.load_from_df(
    svd_ratings[
        ["userId", "movieId", "rating"]
    ],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [28]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("✅ SVD trained on mapped TMDB IDs")

✅ SVD trained on mapped TMDB IDs


In [29]:
test_df = pd.DataFrame(
    testset,
    columns=["userId", "movieId", "actual_rating"]
)

In [30]:
svd_metrics = evaluate_svd_at_k(
    svd_model,
    test_df,
    k=10,
    threshold=4.0
)

print(svd_metrics)

{'precision@10': np.float64(0.02956081081081081), 'recall@10': np.float64(0.030200801496662), 'f1@10': np.float64(0.02429307771646813)}
